# LogiScan Phase 4: Stabilized Unified Training (Forensic Fixed)

### Stabilization Implemented:
1. **WeightedRandomSampler**: Corrects 30:1 class imbalance.
2. **Metric Monitoring**: Per-class tracking with confusion matrices.
3. **Early Stopping**: Based on Macro-F1.
4. **Pre-Export Validation**: Ensures model integrity before artifact creation.

In [ ]:
import json, torch, torch.nn as nn, numpy as np, os
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
REPORT_DIR = Path("training_reports")
REPORT_DIR.mkdir(exist_ok=True)

In [ ]:
DATA_PATH = "unified_training_data.json"
with open(DATA_PATH) as f: data = json.load(f)

texts = [d["text"] for d in data]
label_list = sorted(list(set(d["fallacy"] for d in data)))
label2id = {l: i for i, l in enumerate(label_list)}
labels = [label2id[d["fallacy"]] for d in data]

# 1. WeightedRandomSampler Setup
label_counts = np.bincount(labels)
class_weights = 1.0 / label_counts
sample_weights = [class_weights[l] for l in labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

X_train, X_val, y_train, y_val = train_test_split(texts, labels, test_size=0.15, stratify=labels)

In [ ]:
class FallacyDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.encodings = tokenizer(texts, truncation=True, padding="max_length", max_length=max_length, return_tensors="pt")
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return {"input_ids": self.encodings["input_ids"][i], "attention_mask": self.encodings["attention_mask"][i], "label": self.labels[i]}

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")
train_loader = DataLoader(FallacyDataset(X_train, y_train, tokenizer), batch_size=16, sampler=sampler)
val_loader = DataLoader(FallacyDataset(X_val, y_val, tokenizer), batch_size=32)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("microsoft/deberta-v3-small", num_labels=len(label_list), id2label={i: l for l, i in label2id.items()}, label2id=label2id).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
best_f1 = 0
for epoch in range(6):
    model.train()
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        optimizer.zero_grad()
        outputs = model(batch["input_ids"].to(DEVICE), attention_mask=batch["attention_mask"].to(DEVICE), labels=batch["label"].to(DEVICE))
        outputs.loss.backward()
        optimizer.step()
    
    # Validation & Per-Class Metrics
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            preds = torch.argmax(model(batch["input_ids"].to(DEVICE), attention_mask=batch["attention_mask"].to(DEVICE)).logits, dim=1).cpu()
            all_preds.extend(preds.numpy()); all_labels.extend(batch["label"].numpy())
    
    # 3. Track per-class metrics
    report = classification_report(all_labels, all_preds, target_names=label_list, output_dict=True)
    # 4. Save Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    np.save(REPORT_DIR / f"cm_epoch_{epoch}.npy", cm)
    
    # 5. Early Stopping (Macro F1)
    f1 = report["macro avg"]["f1-score"]
    print(f"Epoch {epoch+1} Val Macro-F1: {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        # 6. Save metadata
        model.save_pretrained("phase4_stabilized_model")
        with open("phase4_stabilized_model/metadata.json", "w") as f: json.dump({"f1": f1, "label_list": label_list}, f)